# Part II · Limitation L1 — Lagged Exposure
**Task 8** | *Worksheet §Part II, Limitation L1 (25 min)*

## The problem: simultaneity

The original exposure indicator is
$$E_k(t) = \mathbf{1}[m(k,t) > 0]$$
where $m(k,t)$ counts neighbours **jumping at the same frame $t$** as the focal cell is observed.  
The future self-jump window is $(t, t+W]$.

**Issue:** the neighbour's jump and the focal cell's future window are anchored to the same frame.  
There is no guarantee of temporal precedence — both cells could be responding
to the same upstream stimulus simultaneously.

**Fix:** use a **lagged exposure**
$$E_k^{(\tau)}(t) = \mathbf{1}[m(k,\, t - \tau) > 0]$$
For $\tau \geq 1$ the neighbour's jump *strictly precedes* the focal cell's look-ahead window,
providing genuine temporal precedence.

If ERK truly propagates from neighbour to focal cell, we expect $\mathrm{RR}^{(\tau)}$
to peak at a small lag $\tau^*$ and decay toward 1 at large $\tau$.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
OUTPUT_ROOT  = PROJECT_ROOT / 'analysis_outputs'

# Load the node table produced in Block 1 (exp 1, site 1, ERK)
NODES_PATH = OUTPUT_ROOT / 'exp_1_site_1_ERKKTR_ratio' / 'nodes.csv.gz'
nodes = pd.read_csv(NODES_PATH)
print(f'Loaded {len(nodes):,} nodes from {NODES_PATH.name}')
nodes.head(3)

---
## Worked toy example — why $\tau=0$ is ambiguous

Consider a single track where the neighbour jumps at frame $t$ and the focal cell jumps at $t+1$.
At $\tau=0$: the focal cell is labelled *exposed* ($E_k(t)=1$) and *future-jumping* ($\mathcal{F}_k(t)=1$ for $W \geq 1$).  
At $\tau=1$: the lagged exposure at $t+1$ looks at $m(k, t)>0$ — still 1, but now the neighbour's jump
**precedes** the focal cell's future window $(t+1, t+1+W]$. The causal arrow is correctly oriented.

In [ ]:
# Reconstruct a minimal example from one track
sample_track = nodes.loc[nodes['track_id'] == nodes['track_id'].value_counts().index[0]]
sample_track = sample_track.sort_values('Image_Metadata_T').head(10)
print(sample_track[['Image_Metadata_T', 'signal_value', 'jump_event',
                     'neighbor_jump_now', 'future_self_jump']].to_string(index=False))

---
## Step 1 — Construct lagged exposure columns

Within each track, shift `neighbor_jump_now` forward by $\tau$ frames.
This means: "did any of my neighbours jump $\tau$ frames ago?"

In [ ]:
nodes = nodes.sort_values(['track_id', 'Image_Metadata_T']).reset_index(drop=True)

MAX_LAG = 6

for tau in range(1, MAX_LAG + 1):
    # shift(tau) within each track: row at position i gets the value from i-tau
    # i.e. "neighbour jump status tau frames ago"
    nodes[f'lagged_exposure_{tau}'] = (
        nodes.groupby('track_id')['neighbor_jump_now']
             .shift(tau)
             .fillna(False)
             .astype(bool)
    )

print('Lagged exposure columns created:')
print([c for c in nodes.columns if 'lagged' in c])

---
## Step 2 — Compute $\mathrm{RR}^{(\tau)}$ for each lag

$$\mathrm{RR}^{(\tau)} = \frac{\Pr(\mathcal{F}_k(t)=1 \mid E_k^{(\tau)}(t)=1)}{\Pr(\mathcal{F}_k(t)=1 \mid E_k^{(\tau)}(t)=0)}$$

In [ ]:
results = []

for tau in range(MAX_LAG + 1):
    col = 'neighbor_jump_now' if tau == 0 else f'lagged_exposure_{tau}'
    exposed   = nodes[nodes[col] == True]
    unexposed = nodes[nodes[col] == False]

    p_exp   = exposed['future_self_jump'].mean()
    p_unexp = unexposed['future_self_jump'].mean()
    rr = p_exp / p_unexp if p_unexp > 0 else np.nan
    rd = p_exp - p_unexp

    results.append({
        'τ (frames)': tau,
        'τ (min)':    tau * 5,
        'n_exposed':  len(exposed),
        'p_exp':      round(p_exp, 4),
        'p_unexp':    round(p_unexp, 4),
        'RD':         round(rd, 4),
        'RR':         round(rr, 4),
    })

df_lag = pd.DataFrame(results).set_index('τ (frames)')
df_lag

---
## Step 3 — Plot $\mathrm{RR}^{(\tau)}$ vs. lag

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

# Left: RR vs lag
axes[0].plot(df_lag['τ (min)'], df_lag['RR'], 'o-', color='darkorange', lw=2)
axes[0].axhline(1, color='grey', ls='--', lw=0.8, label='RR = 1')
axes[0].set_xlabel('Lag $\\tau$ (min)')
axes[0].set_ylabel('$\\mathrm{RR}^{(\\tau)}$')
axes[0].set_title('Lagged relative risk')
axes[0].legend()

# Right: conditional rates vs lag
axes[1].plot(df_lag['τ (min)'], df_lag['p_exp'],   'o-', label='$p_\\mathrm{exp}$')
axes[1].plot(df_lag['τ (min)'], df_lag['p_unexp'], 's--', label='$p_\\mathrm{unexp}$')
axes[1].set_xlabel('Lag $\\tau$ (min)')
axes[1].set_ylabel('Future jump rate')
axes[1].set_title('Conditional rates vs. lag')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
tau_star = df_lag['RR'].idxmax()
print(f"Peak RR at τ* = {tau_star} frames = {tau_star * 5} min")
print(f"RR(τ=0) = {df_lag.loc[0, 'RR']:.4f}  (original estimator)")
print(f"RR(τ=1) = {df_lag.loc[1, 'RR']:.4f}  (first lagged version)")

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# (a) What does τ* × 5 min suggest about the ERK relay timescale?
# (b) Does RR(τ) plateau or drop below 1 at large lags?
# (c) Is the original RR(0) inflated or deflated relative to RR(1)?


---
## ★ Bonus — Reproduce for a second mutation

Load the node table for site 9 (PIK3CA E545K, if computed in Block 3)
and repeat the lagged analysis. Does the peak lag $\tau^*$ differ between WT and PIK3CA E545K?

In [ ]:
# ── YOUR CODE ────────────────────────────────────────────────────────────────
# Load nodes for site 9 and repeat Steps 1–3.
# Compare the lag curves for WT vs. PIK3CA E545K on the same plot.
